# Order Items Data Cleaning

Problems to Investigate
- Convert shipping_limit_date to datetime
- Validate prices and freight values
- Validate order-item numbering
- Verify all orders and products are valid
- Check for whitespace


In [2]:
import pandas as pd 

items_raw = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
items = items_raw.copy()

print("Items Shape:", items_raw.shape)
print("Working Items Shape:", items.shape)

Items Shape: (112650, 7)
Working Items Shape: (112650, 7)


In [4]:
# Convert shipping_limit_date to datetime

items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")

print("Check conversion:\n", items["shipping_limit_date"].dtypes)

Check conversion:
 datetime64[us]


In [8]:
# Validate prices and freight values

print("Validate prices:", items["price"].min(), ",", items["price"].max())

print("Validate freight values:", items["freight_value"].min(), ",", items["freight_value"].max())

Validate prices: 0.85 , 6735.0
Validate freight values: 0.0 , 409.68


In [9]:
# Validate order-item numbering

# Summarize item numbering for each order
item_counts = (
    items.groupby("order_id")["order_item_id"]
    .agg(["count", "min", "max"])
)

# Check whether numbering starts at 1
print(
    "Orders where numbering does not start at 1:",
    (item_counts["min"] != 1).sum()
)

# Check whether numbering contains gaps
print(
    "Orders where item count differs from maximum item ID:",
    (item_counts["count"] != item_counts["max"]).sum()
)

Orders where numbering does not start at 1: 0
Orders where item count differs from maximum item ID: 0


In [16]:
# Verify Order Ids, Products and Seller Ids are valid

orders = pd.read_csv("../data/processed/olist_orders_cleaned.csv")

# Check whether all order IDs exist in the Orders dataset
unmatched_orders = items[
    ~items["order_id"].isin(orders["order_id"])
]

print("Unmatched Order IDs:", len(unmatched_orders))

# Check whether all product IDs exist in the Products dataset
products = pd.read_csv("../data/raw/olist_products_dataset.csv")

unmatched_products = items[
    ~items["product_id"].isin(products["product_id"])
]

print("Unmatched Product IDs:", len(unmatched_products))

# Check whether all seller IDs exist in the Sellers dataset
sellers = pd.read_csv("../data/processed/olist_sellers_cleaned.csv")

unmatched_sellers = items[
    ~items["seller_id"].isin(sellers["seller_id"])
]

print("Unmatched Seller IDs:", len(unmatched_sellers))

Unmatched Order IDs: 0
Unmatched Product IDs: 0
Unmatched Seller IDs: 0


In [14]:
# Check for and remove any whitespace

string_columns = items.select_dtypes(
    include=["object", "str"]
).columns

# Count values with leading or trailing whitespace
for column in string_columns:
    whitespace_count = items[column].str.contains(
        r"^\s|\s$", na=False
    ).sum()

    print(f"{column}: {whitespace_count} values with whitespace")

# Check for leading or trailing whitespace in column names
for column in items.columns:
    if column != column.strip():
        print(f"Whitespace found: '{column}'")

order_id: 0 values with whitespace
product_id: 0 values with whitespace
seller_id: 0 values with whitespace


In [17]:
items.to_csv(
    "../data/processed/olist_order_items_cleaned.csv",
    index=False
)